In [2]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize
import matplotlib.pyplot as plt
import warnings


In [ ]:
# Configuration
np.random.seed(42)  # Reproducibility
plt.style.use('seaborn-v0_8-whitegrid')
warnings.filterwarnings('ignore')

# Population and time parameters
N_PHILIPPINES = 108_000_000  # 2020 census
START_DATE = '2020-03-01'
END_DATE = '2021-12-31'
DT = 1.0  # day

print(f"Analysis: Philippine SIR Model ({START_DATE} to {END_DATE})")
print(f"Population: {N_PHILIPPINES:,}")
print(f"Step size: {DT} day")

In [ ]:
def load_covid_data():
    """
    Load Philippine COVID-19 data from JHU CSSE repository.

    Returns:
        dates: array of datetime objects
        cases: array of cumulative confirmed cases
    """
    # JHU CSSE time series URL
    url = ("https://raw.githubusercontent.com/CSSEGISandData/COVID-19/"
           "master/csse_covid_19_data/csse_covid_19_time_series/"
           "time_series_covid19_confirmed_global.csv")

    # url= ("/content/time_series_covid19_confirmed_global.csv")

    # Load data
    df = pd.read_csv(url)

    # Extract Philippines data
    philippines = df[df['Country/Region'] == 'Philippines'].iloc[0]

    # Drop non-date columns
    date_cols = [c for c in df.columns if c not in
                 ['Province/State', 'Country/Region', 'Lat', 'Long']]

    # Parse dates and values
    dates = pd.to_datetime(date_cols)
    cases = philippines[date_cols].values.astype(float)

    # Filter to study period
    start_dt = pd.to_datetime(START_DATE)
    end_dt = pd.to_datetime(END_DATE)
    mask = (dates >= start_dt) & (dates <= end_dt)

    dates = dates[mask]
    cases = cases[mask]

    print(f"Loaded {len(cases)} daily observations")
    print(f"Cases range: {cases[0]:.0f} to {cases[-1]:,.0f}")

    return dates.values, cases